In [2]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from gensim.utils import deaccent

from chiricoca.config import setup_style

setup_style()


In [ ]:
elecciones = pd.read_csv(
    "data/presidenciales_2021/Servel_20211121_PRESIDENCIALES_CHILE.csv",
    sep=";",
)
elecciones.head()


In [ ]:
elecciones.columns

In [ ]:
clean_name = lambda x: (
    x.replace(" PARTICULAR", "")
    .replace("NRO ", "N°")
    .replace("LOCAL: 1", "")
    .replace("LOCAL :1", "")
    .replace("LOCAL: 2", "")
    .replace("LOCAL :2", "")
    .replace("LOCAL: 3", "")
    .replace("LOCAL :3", "")
    .replace("LOCAL: 4", "")
    .replace("LOCAL :4", "")
    .replace("LOCAL: 5", "")
    .replace("LOCAL: 6", "")
    .replace("LOCAL: 7", "")
    .replace("LOCAL: 8", "")
    .replace("LOCAL: 9", "")
    .replace("ESC.", "ESCUELA")
    .replace("ESC ", "ESCUELA ")
    .replace("EDUC.", "EDUCACIONAL")
    .replace("EDUC ", "EDUCACIONAL ")
    .replace("POLIV ", "POLIVALENTE ")
    .replace("POLIV.", "POLIVALENTE")
    .replace("U.", "UNIVERSIDAD")
    .strip()
)

elecciones['location_name'] = elecciones['local_nombre'].map(clean_name)
elecciones['location_name']

In [6]:
elecciones['region_nombre'] = elecciones['region_nombre'].str.strip()

In [ ]:
codes = pd.read_excel("data/servel/CUT_2018_v04.xls")
codes


In [ ]:
codes["comuna_nombre"] = codes["Nombre Comuna"].str.upper().map(deaccent)
codes["comuna_nombre"]


In [ ]:
codes["Código Región"].unique()


In [ ]:
elecciones["votos_preliminar_string_strip"] = (
    elecciones["votos_preliminar_string"]
    .str.strip()
    .str.replace(r"^$", "0", regex=True)
    .astype(int)
)

elecciones["votos_preliminar_string_strip"]


In [ ]:
elecciones.groupby("candidato")["votos_preliminar_string_strip"].sum()


In [ ]:
elecciones["comuna_nombre"] = elecciones["comuna_nombre"].str.strip()
elecciones["comuna_nombre"]


In [ ]:
turnout = (
    elecciones.groupby(["comuna_nombre", "candidato"])["votos_preliminar_string_strip"]
    .sum()
    .unstack()
)
turnout.head()


In [ ]:
turnout.columns = list(map(lambda x: x.strip(), turnout.columns))
turnout.columns


In [ ]:
candidatos = [
    "EDUARDO ARTES BRICHETTI",
    "GABRIEL BORIC FONT",
    "JOSE ANTONIO KAST RIST",
    "MARCO ENRIQUEZ-OMINAMI GUMUCIO",
    "FRANCO PARISI FERNANDEZ",
    "YASNA PROVOSTE CAMPILLAY",
    "SEBASTIAN SICHEL RAMIREZ",
]
columnas_votos = candidatos + ["Votos Blancos", "Votos Nulos"]
columnas_votos


In [16]:
turnout = (
    turnout.reset_index()
    .assign(comuna_nombre=lambda x: x["comuna_nombre"].map(deaccent))
    .set_index("comuna_nombre")[columnas_votos]
)


In [ ]:
total_votos = turnout[columnas_votos].sum(axis=1)
total_votos


In [ ]:
turnout.sum()


In [ ]:
plebiscito = pd.read_excel(
    "data/plebiscito_2020/Resultados Plebiscito Constitucion Politica 2020.xlsx",
    sheet_name="Chile",
    engine="openpyxl",
).dropna(subset='Local')

plebiscito.head()


In [ ]:
plebiscito.columns


In [21]:
plebiscito.columns = [
    "region_id",
    "region_nombre",
    "provincia_nombre",
    "circsena_nombre",
    "distrito_nombre",
    "comuna_nombre",
    "circele_nombre",
    "local_nombre",
    "mesa_id",
    "tipomesa",
    "mesas_fusionadas",
    "electo",
    "nvoto",
    "opcion_constitucion",
    "votos_tricel",
]


In [22]:
plebiscito['location_name'] = plebiscito['local_nombre'].map(clean_name)

In [ ]:
plebiscito.groupby("opcion_constitucion")["votos_tricel"].sum()


In [24]:
constitucion = [
    "APRUEBO",
    "RECHAZO",
]
columnas_votos_plebiscito = constitucion + ["VOTOS EN BLANCO", "VOTOS NULOS"]


In [25]:
import geopandas as gpd
grid = gpd.read_parquet('results/07_grid.parquet').to_crs('epsg:4326')

In [ ]:
grid_centroids = grid.set_index('h3_cell_id').centroid
#grid_centroids

In [27]:
#grid_centroids.to_file(f'../reports/scl-grid-centroids.geo.json', driver='GeoJSON')

In [ ]:
locations_in_grid = gpd.read_parquet('results/07_locations_in_grid.parquet')
locations_in_grid

In [29]:
elecciones_region = elecciones[elecciones['region_nombre'] == 'METROPOLITANA DE SANTIAGO'].copy()

In [ ]:
elecciones_geoloc = elecciones_region.join(
    locations_in_grid.set_index("location")["h3_cell_id"],
    on="location_name",
    how="inner",
).dropna()
elecciones_geoloc.head()

In [ ]:
from chiricoca.base.weights import normalize_rows

turnout_geoloc = (
    elecciones_geoloc.groupby(["h3_cell_id", "candidato"])[
        "votos_preliminar_string_strip"
    ]
    .sum()
    .unstack()
)
turnout_geoloc.columns = list(map(lambda x: x.strip(), turnout_geoloc.columns))
turnout_geoloc = turnout_geoloc[turnout.columns[:-1]].pipe(normalize_rows)
turnout_geoloc.head()


In [ ]:
ax = grid.join(turnout_geoloc["GABRIEL BORIC FONT"], on="h3_cell_id", how="inner").plot(
    column="GABRIEL BORIC FONT"
)
grid.plot(facecolor="#abacab", alpha=0.2, ax=ax)


In [ ]:
plebiscito_geoloc = plebiscito.join(
    locations_in_grid.set_index("location")["h3_cell_id"],
    on="location_name",
    how="inner",
)
plebiscito_geoloc.head()

In [ ]:
turnout_plebiscito_geoloc = (
    plebiscito_geoloc.groupby(["h3_cell_id", "opcion_constitucion"])["votos_tricel"]
    .sum()
    .unstack(fill_value=0)
    .pipe(normalize_rows)
)
# turnout_plebiscito_geoloc.columns = list(map(lambda x: x.strip(), turnout_geoloc.columns))
# turnout_plebiscito_geoloc = turnout_geoloc[turnout.columns[:-1]].pipe(normalize_rows)
turnout_plebiscito_geoloc.head()

In [ ]:
ax = grid.join(turnout_plebiscito_geoloc["APRUEBO"], on="h3_cell_id", how="inner").plot(
    column="APRUEBO"
)
grid.plot(facecolor="#abacab", alpha=0.2, ax=ax)

In [ ]:
diff = turnout_geoloc.join(turnout_plebiscito_geoloc).assign(diff=lambda x: x['GABRIEL BORIC FONT'] - x['APRUEBO'])['diff']
diff.head()

In [ ]:
from chiricoca.maps import choropleth_map

ax, map_data = choropleth_map(
    grid.join(
        diff,
        on="h3_cell_id",
        how="inner",
    ),
    "diff",
    k=5,
    edgecolor="none",
    alpha=0.75,
    binning="fisher_jenks",
    cbar_args=dict(
        label="Disminuye Votación\nRespecto a Plebiscito",
        height="22%",
        width="2%",
        orientation="vertical",
        location="center right",
        label_size="small",
        bbox_to_anchor=(0.0, 0.0, 0.9, 1.0),
    ),
)

ax.set_title(
    "Caracterización de Votación en Presidenciales 2021 en relación a Plebiscito 2020",
    loc="left",
)

In [ ]:
grid.join(turnout_geoloc.join(turnout_plebiscito_geoloc), on='h3_cell_id')

In [39]:
plt.style.use('dark_background')

In [ ]:
from chiricoca.maps import bivariate_choropleth_map
from chiricoca.maps.utils import add_basemap

ax, palette, cbar_ax = bivariate_choropleth_map(
    grid.join(
        turnout_geoloc.join(turnout_plebiscito_geoloc).dropna(),
        on="h3_cell_id",
        how="inner",
    ),
    "GABRIEL BORIC FONT",
    "APRUEBO",
    palette='RdBu_r',
    edgecolor="none",
    cbar_args=dict(
        location="upper left", bbox_to_anchor=[0.1, 0.0, 0.8, 1.0], width="15%"
    ),
)

add_basemap(ax, "data/scl_network_basemap.tif", grid)